In [10]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split


# Transformer that applies a fixed vector of length-scales
class LengthScaleScaler(BaseEstimator, TransformerMixin):
    def __init__(self, lengthscales=None):
        self.lengthscales = lengthscales

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X / self.lengthscales


# Generate 50D ARD "ball" data and return threshold
def make_ard_ball_data(n_samples=5000, n_dims=50, seed=42):
    rng = np.random.RandomState(seed)
    ls_true = rng.uniform(0.5, 2.0, size=n_dims)
    center = rng.randn(n_dims)
    X = rng.randn(n_samples, n_dims)
    D2 = np.sum(((X - center) ** 2) / (ls_true**2), axis=1)
    thresh = np.median(D2)
    y = (D2 <= thresh).astype(int)
    return X, y, ls_true, thresh


# Helper to train & evaluate for given hyperparameters
def score_svm(lengthscales, C, gamma, X_train, X_val, y_train, y_val):
    pipe = Pipeline(
        [
            ("scale", LengthScaleScaler(lengthscales=lengthscales)),
            ("svc", SVC(kernel="rbf", C=C, gamma=gamma)),
        ]
    )
    pipe.fit(X_train, y_train)
    return pipe.score(X_val, y_val)


# Create data
X, y, ls_true, thresh = make_ard_ball_data(seed=0)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

# Choose true hyperparameters in line with data scale
gamma_true = 1.0 / thresh  # ~1 / (median squared distance)
C_true = 1000.0  # large to approximate hard margin

# Evaluate true hyperparameters
true_acc = score_svm(ls_true, C_true, gamma_true, X_train, X_val, y_train, y_val)

# Random search over 50 trials
rng = np.random.RandomState(1)
random_results = []
for i in range(250):
    ls_rand = rng.uniform(0.5, 2.0, size=50)
    C_rand = 10 ** rng.uniform(-1, 3)  # [0.1, 1000]
    gamma_rand = 10 ** rng.uniform(-3, 0)  # [0.001, 1]
    acc = score_svm(ls_rand, C_rand, gamma_rand, X_train, X_val, y_train, y_val)
    random_results.append(
        {"trial": i + 1, "accuracy": acc, "C": C_rand, "gamma": gamma_rand}
    )

# Summarize
best = max(random_results, key=lambda r: r["accuracy"])
mean_acc = np.mean([r["accuracy"] for r in random_results])

print(f"True hyperparams   → acc = {true_acc:.4f} (C={C_true:.1f}, γ={gamma_true:.4f})")
print(
    f"Best random search → acc = {best['accuracy']:.4f}  (trial {best['trial']}, C={best['C']:.3g}, γ={best['gamma']:.3g})"
)
print(f"Mean random acc    → acc = {mean_acc:.4f}\n")

print("All random-search accuracies:")
for r in random_results:
    print(f" Trial {r['trial']:02d}: acc = {r['accuracy']:.4f}")

True hyperparams   → acc = 0.9660 (C=1000.0, γ=0.0114)
Best random search → acc = 0.9080  (trial 186, C=197, γ=0.00732)
Mean random acc    → acc = 0.7370

All random-search accuracies:
 Trial 01: acc = 0.4990
 Trial 02: acc = 0.8560
 Trial 03: acc = 0.6810
 Trial 04: acc = 0.8390
 Trial 05: acc = 0.8190
 Trial 06: acc = 0.4990
 Trial 07: acc = 0.8450
 Trial 08: acc = 0.8820
 Trial 09: acc = 0.8880
 Trial 10: acc = 0.4990
 Trial 11: acc = 0.4990
 Trial 12: acc = 0.6880
 Trial 13: acc = 0.4990
 Trial 14: acc = 0.8640
 Trial 15: acc = 0.8720
 Trial 16: acc = 0.8300
 Trial 17: acc = 0.8700
 Trial 18: acc = 0.4990
 Trial 19: acc = 0.8550
 Trial 20: acc = 0.4990
 Trial 21: acc = 0.4990
 Trial 22: acc = 0.7540
 Trial 23: acc = 0.7760
 Trial 24: acc = 0.4990
 Trial 25: acc = 0.8600
 Trial 26: acc = 0.4990
 Trial 27: acc = 0.8320
 Trial 28: acc = 0.8880
 Trial 29: acc = 0.4980
 Trial 30: acc = 0.7060
 Trial 31: acc = 0.8000
 Trial 32: acc = 0.4990
 Trial 33: acc = 0.4990
 Trial 34: acc = 0.8250

In [73]:
from ucimlrepo import fetch_ucirepo
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split


class LengthScaleScaler(BaseEstimator, TransformerMixin):
    def __init__(self, lengthscales=None):
        self.lengthscales = lengthscales

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X / self.lengthscales


def score_svm(lengthscales, C, gamma, X_train, X_val, y_train, y_val):
    pipe = Pipeline(
        [
            ("scale", LengthScaleScaler(lengthscales=lengthscales)),
            ("svc", SVC(kernel="rbf", C=C, gamma=gamma)),
        ]
    )
    pipe.fit(X_train, y_train)
    return pipe.score(X_val, y_val)


# fetch dataset
from ucimlrepo import fetch_ucirepo

# fetch dataset
statlog_landsat_satellite = fetch_ucirepo(id=146)

# data (as pandas dataframes)
X = statlog_landsat_satellite.data.features
y = statlog_landsat_satellite.data.targets
y = y.values.ravel()
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)
rng = np.random.RandomState(1)
random_results = []
for i in range(50):
    ls_rand = rng.uniform(0.1, 5.0, size=36)
    C_rand = 10 ** rng.uniform(-1, 3)  # [0.1, 1000]
    gamma_rand = 10 ** rng.uniform(-4, 0)  # [0.001, 1]
    acc = score_svm(ls_rand, C_rand, gamma_rand, X_train, X_val, y_train, y_val)
    random_results.append(
        {"trial": i + 1, "accuracy": acc, "C": C_rand, "gamma": gamma_rand}
    )

# Summarize
best = max(random_results, key=lambda r: r["accuracy"])
mean_acc = np.mean([r["accuracy"] for r in random_results])

print(
    f"Best random search → acc = {best['accuracy']:.4f}  (trial {best['trial']}, C={best['C']:.3g}, γ={best['gamma']:.3g})"
)
print(f"Mean random acc    → acc = {mean_acc:.4f}\n")
print("All random-search accuracies:")
for r in random_results:
    print(f" Trial {r['trial']:02d}: acc = {r['accuracy']:.4f}")

Best random search → acc = 0.9099  (trial 17, C=9.47, γ=0.000338)
Mean random acc    → acc = 0.5050

All random-search accuracies:
 Trial 01: acc = 0.2214
 Trial 02: acc = 0.9075
 Trial 03: acc = 0.3077
 Trial 04: acc = 0.2214
 Trial 05: acc = 0.2214
 Trial 06: acc = 0.2214
 Trial 07: acc = 0.8788
 Trial 08: acc = 0.6923
 Trial 09: acc = 0.2214
 Trial 10: acc = 0.8998
 Trial 11: acc = 0.7653
 Trial 12: acc = 0.2238
 Trial 13: acc = 0.8943
 Trial 14: acc = 0.8974
 Trial 15: acc = 0.8951
 Trial 16: acc = 0.5789
 Trial 17: acc = 0.9099
 Trial 18: acc = 0.2214
 Trial 19: acc = 0.2214
 Trial 20: acc = 0.5952
 Trial 21: acc = 0.6535
 Trial 22: acc = 0.2214
 Trial 23: acc = 0.9037
 Trial 24: acc = 0.9060
 Trial 25: acc = 0.6589
 Trial 26: acc = 0.9060
 Trial 27: acc = 0.8928
 Trial 28: acc = 0.2214
 Trial 29: acc = 0.2214
 Trial 30: acc = 0.2284
 Trial 31: acc = 0.2214
 Trial 32: acc = 0.2766
 Trial 33: acc = 0.6371
 Trial 34: acc = 0.2214
 Trial 35: acc = 0.9091
 Trial 36: acc = 0.2246
 Tria

In [69]:
import time

start = time.time()
print(
    score_svm(
        rng.uniform(0.1, 5.0, size=36), 140, 0.00293, X_train, X_val, y_train, y_val
    )
)
end = time.time()
print(end - start)

0.585081585081585
1.5094108581542969


In [43]:
X_train

,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,...,Attribute608,Attribute609,Attribute610,Attribute611,Attribute612,Attribute613,Attribute614,Attribute615,Attribute616,Attribute617
3149,-0.2842,0.2024,0.6564,0.6238,-0.0552,-0.3784,-0.4396,-0.4764,-0.6074,-0.5338,...,0.0598,0.0598,-0.0940,-0.2650,0.0598,0.1282,0.1282,0.0256,-0.0598,-0.2650
3083,-0.4224,-0.0902,0.1768,0.5306,0.4512,-0.3718,-0.2924,-0.3936,-0.4080,-0.3286,...,0.5704,0.5974,0.5034,0.3692,0.5974,0.6376,0.5034,0.4362,0.5168,0.2484
4235,-0.4706,-0.0672,0.2268,0.8152,1.0000,0.9664,0.4622,0.0588,-0.0252,0.0840,...,0.9774,0.6046,0.5706,0.6158,0.5594,0.5368,0.7062,0.6384,0.2656,0.0960
1362,-0.4542,0.1584,0.1876,0.4500,0.4708,0.8334,0.8834,0.8792,0.7542,0.5958,...,0.8462,0.7384,0.9076,0.9384,0.8770,0.7846,0.7846,0.7846,0.5384,0.2462
2611,-0.6404,-0.2644,0.1226,0.6676,0.7276,0.5258,-0.0028,-0.0246,-0.2644,-0.1662,...,1.0000,0.9672,0.6558,0.9344,0.8360,0.6230,0.6886,0.8524,0.8524,0.5410
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4931,-0.2666,0.1192,0.3088,0.4176,0.6562,0.8562,0.8246,0.8000,0.9264,0.9158,...,-0.1052,-0.3334,0.1404,-0.0176,0.0000,-0.0702,-0.2106,-0.1228,-0.1754,-0.7544
3264,-0.2232,0.2830,0.3396,0.4622,0.9026,0.9402,1.0000,0.9466,0.9812,0.8428,...,0.0164,-0.0820,0.1148,0.5082,0.4426,0.4426,0.3442,0.0492,-0.3606,-0.4426
1653,-0.3668,-0.1598,-0.1924,-0.7218,-0.8314,-0.8284,-0.9498,-0.8964,-0.8106,-0.8136,...,0.3030,0.8182,0.3636,0.3334,0.4546,0.3636,0.1212,-0.0304,0.0910,-0.2122
2607,-0.5168,0.2428,0.4062,0.3076,-0.2860,-0.5938,-0.7068,-0.8630,-0.9134,-0.8462,...,0.2942,0.2942,0.0352,0.1764,0.1294,0.1764,0.1294,0.2706,0.1530,-0.3412
